## `POPSRegressionEllipse`: ellipsoid posteriors by direct optimization

Comparing `BayesianRidge` (epistemic only), `POPSRegression`
(sampling-based POPS hypercube) and `POPSRegressionEllipse`
(uniform-ellipsoid posterior fit by direct minimization of the
generalization-error objective) on a misspecified polynomial surrogate
fit to low-noise data.

The ellipsoid bounds track the POPS hypercube bounds, but are obtained by
an interior-point optimization of the exact projected-ball pushforward
likelihood — no posterior sampling is involved.

In [ ]:
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
from popsregression import POPSRegression, POPSRegressionEllipse

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def target_function(x):
    return (x**3 + 0.01 * x**4) * 0.1 + np.sin(x) * x * 10.0


def generate_data(N):
    x_train = np.sort(
        np.append(np.random.uniform(-1, 1, N), np.linspace(-1, 1, 2)) * 10
    )
    x_dense = np.linspace(-1.1, 1.1, 51) * 10
    y_dense = target_function(x_dense)

    p = PolynomialFeatures(degree=4, include_bias=True)
    X_train = p.fit_transform(x_train.reshape(-1, 1))
    X_dense = p.fit_transform(x_dense.reshape(-1, 1))
    y_train = target_function(x_train)
    return X_train, x_train, y_train, X_dense, x_dense, y_dense


def plot_panel(ax, x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=None, y_min=None):
    if y_max is not None and y_min is not None:
        ax.fill_between(x_dense, y_min, y_max, alpha=0.2, facecolor="0.5",
                        label="max/min")
    else:
        ax.fill_between(x_dense, y_pred - 4 * y_std, y_pred + 4 * y_std,
                        alpha=0.2, facecolor="0.5", label=r"$\pm4\sigma$")
    ax.fill_between(x_dense, y_pred - 2 * y_std, y_pred + 2 * y_std,
                    alpha=0.5, facecolor="C1", label=r"$\pm2\sigma$")
    ax.plot(x_dense, y_pred, "C1-", lw=4)
    ax.plot(x_train, y_train, "b.", label="Train")
    ax.plot(x_dense, y_dense, "k-")

### BayesianRidge vs POPS Hypercube vs POPS Ellipse

Fitting a quartic polynomial (P=5) to a complex oscillatory function at
N = 10, 50, 500 training points. BayesianRidge epistemic uncertainty
vanishes with more data; both POPS variants maintain honest uncertainty
where the polynomial deviates from the truth. The ellipse bounds follow
the hypercube bounds while being produced by direct optimization.

In [ ]:
np.random.seed(42)

titles = ["Bayesian Ridge", "POPS Hypercube", "POPS Ellipse"]
fig, axs = plt.subplots(3, 3, figsize=(8, 7), sharex=True, sharey=True)
N_array = [10, 50, 500]

for i, N in enumerate(N_array):
    X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(N)

    bay = BayesianRidge(fit_intercept=False)
    hyc = POPSRegression(leverage_percentile=0.0, posterior="hypercube")
    ell = POPSRegressionEllipse(random_state=0)

    bay.fit(X_train, y_train)
    hyc.fit(X_train, y_train)
    ell.fit(X_train, y_train)

    # BayesianRidge - epistemic only (no aleatoric alpha_)
    b_pred = bay.predict(X_dense, return_std=False)
    b_std = np.sqrt(np.sum(np.dot(X_dense, bay.sigma_) * X_dense, axis=1))
    plot_panel(axs[0, i], x_dense, y_dense, x_train, y_train, b_pred, b_std)

    # POPS Hypercube (sampling-based)
    y_pred, y_std, y_max, y_min = hyc.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[1, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    # POPS Ellipse (direct optimization; bounds = pushforward support)
    y_pred, y_std, y_max, y_min = ell.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[2, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    axs[0, i].set_title(f"N = {N}")

for j, title in enumerate(titles):
    axs[j, 0].set_ylim(-250, 250)
    axs[j, 0].set_xlim(-10, 10)
    axs[j, 1].legend(fontsize=9, loc="lower center")
    axs[j, 0].set_ylabel(title, fontsize=13)

plt.tight_layout()
plt.show()

### Observations

- **N/P = 2**: all methods show wide uncertainty; both POPS variants
  cover the true function (black line).
- **N/P = 10 and 100**: BayesianRidge epistemic uncertainty collapses,
  while the hypercube and ellipse bounds persist, reflecting the
  structural inability of the quartic to match the oscillatory target.
- The **ellipse** row tracks the **hypercube** row closely, but its
  bounds come from the optimized ellipsoid support
  $\text{mean} \pm \sqrt{x^\top B x + \delta^2}$ rather than from
  posterior samples, and every training point is covered by
  construction (`coverage_fraction_ == 1`).

### Closed-form PAC-Bayes layer

With `pac_bayes=True` the fit adds a Gaussian hyperprior ridge and a
diagonal Laplace hyperposterior, giving closed-form KL and PAC-bound
components — still without any sampling. The predictive uncertainty
gains the analytic hyperposterior spread.

The layer is most informative in the **low-data regime**: at N/P ~ 2 the
Laplace covariance is wide (the ellipsoid itself is uncertain), the
`kl_/N` term dominates the bound, and the predictive widths are visibly
inflated. As N grows the hyperposterior concentrates at rate N and
`bound_` tightens toward the empirical error. `update_hyperprior=True`
(Tipping/MacKay evidence update of the hyperprior scale `tau2`) is the
recommended companion — a fixed `hyperprior_scale` that is too small
over-regularizes large-N fits, while the evidence update adapts it.

In [ ]:
for N in [10, 50, 500]:
    X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(N)
    pac = POPSRegressionEllipse(
        random_state=0, pac_bayes=True, update_hyperprior=True
    )
    pac.fit(X_train, y_train)
    d = pac.hyper_sigma_diag_.size
    print(
        f"N={N:<4d} coverage={pac.coverage_fraction_:.2f}  "
        f"G_hat={pac.objective_:6.3f}  H={pac.empirical_H_:6.3f}  "
        f"kl/N={pac.kl_ / len(y_train):5.2f}  bound={pac.bound_:6.3f}  "
        f"tau2={pac.tau2_:6.3g}  gamma={pac.gamma_:5.1f} of {d}"
    )

The bound tightens monotonically with N while every training point stays
covered: the misspecification width in `B` is retained, and the extra
hyperposterior spread — the epistemic uncertainty *about the ellipsoid*
itself — decays at rate N, exactly the behaviour the hierarchical
PAC-Bayes construction prescribes.